# 超参数搜索

网格搜索GridSearchCV 和 随机搜索RandomizedSearchCV 都是 sklearn 中用于自动搜索模型超参数的工具。它们最大的区别是：
- GridSearchCV：把你指定的参数组合全部尝试一遍。把指定网格里的每一种参数组合都评估一遍。适合参数范围小、希望完整比较候选组合的场景。
- RandomizedSearchCV：从参数搜索空间中随机抽取指定数量的组合进行尝试。只抽取指定数量的参数组合进行评估。适合参数范围大、计算预算有限的场景。

## 网格搜索GridSearchCV 和 随机搜索RandomizedSearchCV 对比
| 对比项              | GridSearchCV       | RandomizedSearchCV |
| ---------------- | ------------------ | ------------------ |
| 搜索方式             | 遍历全部候选组合           | 随机抽取组合             |
| 搜索次数             | 由候选组合数量决定          | 由 `n_iter` 决定      |
| 是否一定尝试全部组合       | 是                  | 否                  |
| 计算成本             | 组合越多，成本越高          | 可以控制搜索预算           |
| 结果是否一定是指定网格中的最优解 | 是，前提是比较结果正常且评分规则固定 | 否，只能保证是已抽样组合中的最佳结果 |
| 是否支持交叉验证         | 支持                 | 支持                 |
| 常用场景             | 精细调参、小型搜索空间        | 大型搜索空间、先粗调再精调      |


## 为什么要使用 bGridSearchCV 和 RandomizedSearchCV  超参数搜索方法？
- 比如现在有个模型有3个参数，每一个参数又有多个取值（如分别有5、6和7种取值），如果不使用 SearchCV 搜索方法也可以完成相关功能，那么就要实现一个3层for来遍历训练模型，以此来找到最优的模型超参数。
- 此时，如果使用 *SearchCV 方法来搜索超参数就变得很简单了，由内部自动实现超参数的搜索



## 机器学习调参的标准流程

1. 准备数据

   检查缺失值、标签分布、特征含义，并确认没有把目标变量或未来信息混入特征。
2. 先划分训练集和测试集

   测试集在选参数的过程中保持独立。
3. 定义模型和参数网格

   从少量关键参数开始，不要一开始就把所有参数都放入网格。
4. 选择合适的评分指标和交叉验证策略

   分类关注 Accuracy、Precision、Recall、F1 等；回归关注 MAE、RMSE、R² 等。
5. 执行 GridSearchCV

   调用 `fit()`，让搜索器比较参数组合。
6. 分析 `cv_results_`

   查看平均分、标准差、排名和耗时，不只看最佳分数。
7. 使用独立测试集评估

   调用 `predict()`，计算测试集指标，并与调参前的基线模型比较。
8. 保存模型并记录参数

   保存最佳 Pipeline 或模型，以及特征处理逻辑、评分指标和数据版本。

如果你后续希望进一步学习，可以重点掌握三个方向：Pipeline 嵌套参数搜索、多指标评估，以及 GridSearchCV 与 RandomizedSearchCV 的对比实验。



[超参数调优完整指南](https://scikit-learn.org/stable/modules/grid_search.html)


In [ ]:
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV

In [ ]:
param_grid_tmp = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 10, 20],
    "min_samples_split": [2, 5]
}

# GridSearchCV
# 会尝试 3*3*2=18次搜索
# 如果设置cv=5，即进行5折交叉运算，那么将进行3*3*2*5=90次交叉验证模型训练验证，搜索结束后的参数是所有参数组合中最优的，也并不一定是全局最优的

# RandomizedSearchCV
# 只尝试指定次数，例如设置 n_iter=5、cv=5：
# 那么进行n_iter=5次参数搜索，总共搜索5*5=25次参数搜索

# 有一个区别：GridSearchCV 通常使用离散的候选值，而 RandomizedSearchCV 还可以使用连续概率分布，搜索更广泛的数值范围。


# GridSearchCV：遍历全部组合
# param_grid=param_grid

# RandomizedSearchCV：随机抽取指定数量的组合
# param_distributions=param_distributions
# n_iter=20

 [网格搜索 GridSearchCV](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html)






### GridSearchCV 的工作流程

1\. 定义候选参数

例如 C = 0.01、0.1、1、10

2\. 生成参数组合

每组参数都需要单独训练和验证

3\. 执行交叉验证

默认 cv=5，即通常进行 5 折交叉验证

4\. 比较各组平均得分

例如比较 accuracy、F1、R² 等指标

5\. 找到最佳参数并重新训练

通过 best_params\_、best_score\_、best_estimator\_ 获取结果

需要记住：GridSearchCV 不负责替你选择所有模型，也不是直接修改模型参数。它是在你指定的候选范围内，寻找交叉验证得分最高的参数组合。

### 模型参数和超参数不是一回事。

| 类型           | 含义                | 例子                            |
| ------------ | ----------------- | ----------------------------- |
| 模型参数         | 模型训练过程中从数据中学习到的内容 | 线性回归的权重、偏置                    |
| 超参数          | 训练前由开发者设置的配置      | 逻辑回归的 `C`、随机森林的 `max_depth`   |
| GridSearchCV | 自动比较多种超参数配置       | 找出验证效果较好的 `C` 和 `max_iter` 组合 |




### GridSearchCV 使用方法


```python
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    verbose=1,
    refit=True
)
```

| 参数                   | 作用               | 常见设置                           |
| -------------------- | ---------------- | ------------------------------ |
| `estimator`          | 要调参的模型或 Pipeline | `LogisticRegression()`         |
| `param_grid`         | 候选超参数及其取值        | 字典或字典列表                        |
| `scoring`            | 用什么指标评估          | `accuracy`、`f1`、`roc_auc`、`r2` |
| `cv`                 | 交叉验证的折数或划分器      | `5`、`10`                       |
| `n_jobs`             | 并行任务数量           | `-1` 表示使用所有可用处理器               |
| `verbose`            | 输出训练过程信息的详细程度    | `0`、`1`、`2`                    |
| `refit`              | 搜索完成后是否用最佳参数重新训练 | 通常使用 `True`                    |
| `return_train_score` | 是否保存每组参数的训练集得分   | 默认 `False`                     |
| `pre_dispatch`       | 并行任务预先分发的数量      | 默认 `"2*n_jobs"`                |
| `error_score`        | 某组参数训练失败时如何处理    | 默认 `np.nan`，调试时可用 `"raise"`    |

对于 scoring 参数，
- 分类器通常默认使用准确率 accuracy。
- 回归器通常默认使用决定系数 \(R^2\)。




In [ ]:
# 使用 GridSearchCV 对心脏疾病预测进行超参数搜索

# 导包
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1、加载数据
df = pd.read_csv("../../../data/ml/heart-disease.csv")

# 2、数据拆分
X_src = df.drop(["target"], axis=1)
y_src = df["target"]
print(X_src.shape, y_src.shape)

# 3、数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print(X_scaled.shape, y_src.shape)

# 5、数据划分
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_src, test_size=0.2, random_state=23, shuffle=True)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# 6、模型定义
model = LogisticRegression()

# 7、GridSearchCV 定义
# 定义要搜索的超参数
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 1],
    "solver": ["liblinear", "saga"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=23 )
grid_search = GridSearchCV(
    estimator=model,
    param_grid=param_grid,
    cv=cv,
    scoring="accuracy",
    n_jobs=2,
    verbose=1,
    refit=True
)

# 8、训练模型
grid_search.fit(X_train, y_train)

print("最佳参数：", grid_search.best_params_)
print("最佳交叉验证 ROC-AUC：", grid_search.best_score_)
# 实际学习中，你还应该检查所有参数组合的得分，了解参数改变后模型效果如何变化。GridSearchCV 把这些结果保存在 cv_results_ 中
results = pd.DataFrame(grid_search.cv_results_)
print(results.head())

# 取出最好的那个模型
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

"""
在 Pipeline 中使用
数名要写成 model__C？
因为 Pipeline 中模型的名称是 model，而 C 是这个模型的参数。
中间的双下划线 __ 表示访问 Pipeline 内部某个步骤的参数。

# 3. 定义 Pipeline
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        random_state=23
    ))
])

# 4. 定义参数网格
param_grid = {
    "model__C": [0.01, 0.1, 1, 10, 100],
    "model__solver": ["lbfgs", "liblinear"]
}


# 使用字典列表，因为有些参数组合不适用，所以使用字典列表来分开组合搜索
param_grid = [
    {
        "kernel": ["linear"],
        "C": [0.1, 1, 10]
    },
    {
        "kernel": ["rbf"],
        "C": [0.1, 1, 10],
        "gamma": ["scale", 0.01, 0.1]
    }
]

"""

详细讲讲 RandomizedSearchCV 的使用，完整的使用教程

In [ ]:

import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# 1、加载数据
df = pd.read_csv("../../../data/ml/heart-disease.csv")

# 2、数据拆分
X_src = df.drop(["target"], axis=1)
y_src = df["target"]
print(X_src.shape, y_src.shape)

# 3、数据标准化
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_imputed)
print(X_scaled.shape, y_src.shape)

# 5、数据划分
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y_src, test_size=0.2, random_state=23, shuffle=True)
print(X_train.shape, y_train.shape, X_test.shape, y_test.shape)

# 6、模型定义
model = LogisticRegression()

# 7、GridSearchCV 定义
# 定义要搜索的超参数
param_grid = {
    "C": [0.01, 0.1, 1, 10, 100],
    "l1_ratio": [0, 1],
    "solver": ["liblinear", "saga"],
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=23 )

"""
参数	含义	常见用法
estimator	要调参的模型或 Pipeline	RandomForestClassifier()
param_distributions	参数候选列表或分布	字典或字典列表
n_iter	随机抽取多少组参数	10、20、50
scoring	用什么指标比较参数组合	"accuracy"、"f1"、"r2"
cv	交叉验证策略	5、10、StratifiedKFold(...)
n_jobs	并行任务数量	-1 使用所有可用处理器
random_state	控制随机抽样的可复现性	23
verbose	输出多少搜索过程信息	0、1、2
refit	是否用最佳参数重新训练模型	通常为 True
return_train_score	是否保存训练折的评分	True 或 False
error_score	候选参数训练失败时如何处理	默认 np.nan，调试可用 "raise"
pre_dispatch	并行任务预先分发的数量	默认 "2*n_jobs"
"""
grid_search = RandomizedSearchCV(
    estimator=model,
    param_distributions=param_grid,
    n_iter=20,
    scoring="accuracy",
    cv=5,
    n_jobs=-1,
    random_state=23,
    verbose=1,
    refit=True
)

# 8、训练模型
grid_search.fit(X_train, y_train)

print("最佳参数：", grid_search.best_params_)
print("最佳交叉验证 ROC-AUC：", grid_search.best_score_)
# 实际学习中，你还应该检查所有参数组合的得分，了解参数改变后模型效果如何变化。GridSearchCV 把这些结果保存在 cv_results_ 中
results = pd.DataFrame(grid_search.cv_results_)
print(results.head())

# 取出最好的那个模型
best_model = grid_search.best_estimator_
y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]
